In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score


In [2]:
# Load Dataset
df = pd.read_csv('IMDb Movies India.csv (3).csv')

In [3]:
df 

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,,NaN,NaN,Drama,NaN,NaN,J.S. Randhawa,Manmauji,Birbal,Rajendra Bhatia
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
2,#Homecoming,(2021),90 min,"Drama, Musical",NaN,NaN,Soumyajit Majumdar,Sayani Gupta,Plabita Borthakur,Roy Angana
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
4,...And Once Again,(2010),105 min,Drama,NaN,NaN,Amol Palekar,Rajat Kapoor,Rituparna Sengupta,Antara Mali
...,...,...,...,...,...,...,...,...,...,...
15504,Zulm Ko Jala Doonga,(1988),NaN,Action,4.6,11,Mahendra Shah,Naseeruddin Shah,Sumeet Saigal,Suparna Anand
15505,Zulmi,(1999),129 min,"Action, Drama",4.5,655,Kuku Kohli,Akshay Kumar,Twinkle Khanna,Aruna Irani
15506,Zulmi Raj,(2005),NaN,Action,NaN,NaN,Kiran Thej,Sangeeta Tiwari,NaN,NaN
15507,Zulmi Shikari,(1988),NaN,Action,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# remove extra spaces from column name
df.columns = df.columns.str.strip()

In [5]:
# data cleaning
# Clean Duration
df["Duration"] = df["Duration"].astype(str).str.replace(" min", "")
df["Duration"] = pd.to_numeric(df["Duration"], errors="coerce")

# Clean Votes
df["Votes"] = df["Votes"].astype(str).str.replace(",", "")
df["Votes"] = pd.to_numeric(df["Votes"], errors="coerce")

# Clean Rating
df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")

# Drop rows with missing target
df = df.dropna(subset=["Rating"])

In [6]:
# Select useful feartures
features = [
    'Genre',
    'Director',
    'Actor 1',
    'Actor 2',
    'Actor 3',
    'Duration',
    'Votes'
]

x = df[features]
y = df['Rating']

In [7]:
# Numerical and Categorical Columns
num_cols = ['Duration','Votes']
cat_cols = ['Genre','Director','Actor 1','Actor 2','Actor 3']

In [8]:
# preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat",
         Pipeline([
             ("imputer", SimpleImputer(strategy="most_frequent")),
             ("encoder", OneHotEncoder(handle_unknown="ignore"))
         ]),
         cat_cols)
    ]
)

In [9]:
# Model
model = RandomForestRegressor(
    n_estimators = 100,
    random_state = 42
)

In [10]:
# Pipeline
pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',model)
])

In [11]:
# Train-Test-Split
X_train,X_test,y_train,y_test=train_test_split(
    x,y,
    test_size = 0.2,
    random_state = 42
)



In [13]:
# Train Model
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['Duration', 'Votes']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Genre', 'Director',
                                                   'Actor 1', 'Actor 2',
                                                   'Actor 3'])])),
                ('model', RandomForestRegressor(random_state=42))])

In [14]:
# Prediction
y_pred = pipeline.predict(X_test)

In [15]:
# Evaluation
mae = mean_absolute_error(y_test,y_pred)
rmse = np.sqrt(mean_squared_error(y_test,y_pred))
r2 = r2_score(y_test,y_pred)

In [16]:
print('MAE :',round(mae,2))
print('RMSE :',round(rmse,2))
print('R2 Score:',round(r2,2))

MAE : 0.9
RMSE : 1.19
R2 Score: 0.24
